# Lab 9: find similar abstracts

In the lecture, we built a TF-IDF table for OpenAlex abstracts and used cosine similarity to find abstracts that use similar language. In this lab, you will run that workflow yourself and judge the matches by reading them.

## The question

Which abstracts in this sample are most similar to each other, and do the matches hold up when you read them?

## What you will practice

By the end of the lab, you should be able to:

- explain what one row of a TF-IDF table represents;
- use cosine similarity to find the abstracts closest to an anchor abstract;
- list the shared terms that explain a similarity score;
- decide, by reading, whether a match makes sense.

Section 5, on sorting abstracts into groups, is optional.

## Set up

Run these cells from top to bottom. If you restart the kernel, start here again.

In [ ]:
from pathlib import Path
import html

import pandas as pd

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
DATA_BASE = Path("data")
if not DATA_BASE.exists():
    DATA_BASE = Path("..") / "data"

In [ ]:
if not DATA_BASE.exists():
    DATA_BASE = "https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/data"

def data_path(filename):
    return DATA_BASE / filename if isinstance(DATA_BASE, Path) else f"{DATA_BASE}/{filename}"

## 1. Read the abstracts

This file contains 80 OpenAlex works with at least one UC Berkeley-affiliated authorship. That does not mean every work was led by Berkeley or written by Berkeley faculty.

In [ ]:
works = pd.read_csv(data_path("openalex_berkeley_abstracts_2024_sample.csv"))
works = works.dropna(subset=["abstract"]).reset_index(drop=True)
works.shape

In [ ]:
works[["title", "primary_domain", "primary_field"]].head()

In [ ]:
works["primary_domain"].value_counts(dropna=False)

Before you compare abstracts, note which domains appear most often. How might that imbalance affect which abstracts end up close to each other?

As we saw in the lecture, a few abstracts contain web markup for equations (`<mml:math ...>`) and codes such as `&amp;`. The next cell removes the tags and converts the codes to ordinary characters. The cleaned text goes in a new column, `text`.

In [ ]:
works["text"] = works["abstract"].str.replace(r"<[^>]+>", " ", regex=True).map(html.unescape)

## 2. Build document vectors

TF-IDF makes a table with one row per abstract and one column per term. Each row is the document vector for one abstract.

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english", min_df=2, max_features=1500)
tfidf_matrix = vectorizer.fit_transform(works["text"])
terms = vectorizer.get_feature_names_out()
tfidf_matrix.shape

Record the shape in your notes. What do the two numbers count?

## 3. Score pairs of abstracts

`cosine_similarity` compares two rows and returns a score. A score of 0 means the two abstracts share no terms in the table. Scores closer to 1 mean they give weight to more of the same terms.

In [ ]:
cosine_similarity(tfidf_matrix[0], tfidf_matrix[3])

Passing the whole table scores every pair at once. Row `i` of the result holds the scores between abstract `i` and every abstract, including itself.

In [ ]:
similarity = cosine_similarity(tfidf_matrix)
similarity.shape

## 4. Find and read the nearest abstracts

Choose one abstract as the anchor. Start with row 3, then change `anchor_id` to 46 or to a work that interests you.

In [ ]:
anchor_id = 3
works.loc[anchor_id, ["title", "primary_domain", "text"]]

Sort the anchor's scores from high to low. The first entry is the anchor itself, so we skip it and keep the next three.

In [ ]:
scores = similarity[anchor_id]
nearest_ids = scores.argsort()[::-1][1:4]

In [ ]:
neighbors = works.loc[nearest_ids, ["title", "primary_domain"]].copy()
neighbors["similarity"] = scores[nearest_ids]
neighbors

The next cell lists the terms that the anchor and its closest neighbor share. Each number is how much that term adds to the similarity score, and the numbers add up to the score.

In [ ]:
anchor_row = tfidf_matrix[anchor_id].toarray()[0]
neighbor_row = tfidf_matrix[nearest_ids[0]].toarray()[0]
shared = pd.Series(anchor_row * neighbor_row, index=terms)
shared[shared > 0].sort_values(ascending=False).head(10)

In [ ]:
works.loc[[anchor_id, nearest_ids[0]], ["title", "text"]]

### Your response

Run section 4 for anchor 3 and for anchor 46. For each one, read the anchor and its closest neighbor. Do the shared terms reflect what the two abstracts are about, or something else, such as how results were reported? Does the match hold up? Write two or three sentences per anchor.

## 5. Optional: sort the abstracts into groups

KMeans puts each abstract into one of `n_clusters` groups so that abstracts in a group have similar vectors. `random_state` fixes the random starting point, so you get the same groups each time you run the notebook. The group numbers are labels, not rankings.

In [ ]:
from sklearn.cluster import KMeans

model = KMeans(n_clusters=5, random_state=211)
works["cluster"] = model.fit_predict(tfidf_matrix)
works["cluster"].value_counts().sort_index()

For each group, the next cell prints the eight terms with the highest average weight.

In [ ]:
for number, center in enumerate(model.cluster_centers_):
    top_terms = terms[center.argsort()[::-1][:8]]
    print(number, ", ".join(top_terms))

In [ ]:
cluster_to_read = 0
works.loc[works["cluster"] == cluster_to_read, ["title", "primary_domain"]]

### Your response

Give the group a short, cautious description based on its top terms and titles. Read at least two abstracts in full. If the group is too mixed to describe, say so and point to the titles that make it difficult.

If you have time, change `n_clusters` to 4, rerun this section, and check whether the abstracts in your group stay together.

## Exit check

Before you leave, make sure your notebook shows:

- the shape of the TF-IDF table;
- the nearest abstracts and shared terms for anchors 3 and 46;
- your written responses about whether the matches hold up.

Restart the kernel and run the whole notebook once. Fix any cell that depends on something you ran out of order.